In [0]:
from pyspark.sql import SparkSession

In [0]:
from pyspark.sql.functions import col,lit,expr,regexp_replace,when 
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType
print('done')

done


In [0]:
spark = SparkSession.builder.appName("Test").getOrCreate()

In [0]:
data = [
    ("101", "Asha",    "South", "Electronics", "1", "45000"),
    ("102", "Ravi",    "North", "Grocery",     "5",  "1200"),
    ("103", "Meena",   "South", "Grocery",     "3",   "900"),
    ("104", "Karthik", "East",  "Electronics", "2", "60000"),
    ("105", "Divya",   "North", "Clothing",    "4",  "3200"),
    ("106", "Imran",   "East",  "Clothing",    "2",  "1800"),
    ("107", "Sneha",   "South", "Electronics", "1", "30000"),
    ("108", "Arjun",   "North", "Electronics", "1", "52000"),
    ("109", "Kavya",   "East",  "Grocery",     "6",  "1500"),
    ("110", "Rahul",   "South", "Clothing",    "3",  "2700"),
]
cols = ["order_id", "customer", "region", "category", "qty", "amount"]

In [0]:
orders = spark.createDataFrame(data,cols)

In [0]:
orders.display()
orders.printSchema()

order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000
102,Ravi,North,Grocery,5,1200
103,Meena,South,Grocery,3,900
104,Karthik,East,Electronics,2,60000
105,Divya,North,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,30000
108,Arjun,North,Electronics,1,52000
109,Kavya,East,Grocery,6,1500
110,Rahul,South,Clothing,3,2700


root
 |-- order_id: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- amount: string (nullable = true)



In [0]:
orders = (orders.withColumn("order_id", col("order_id").cast("int"))
                .withColumn("qty", col("qty").cast("int"))
                .withColumn("amount", col("amount").cast("float")))
orders.display()
orders.printSchema()


order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000.0
102,Ravi,North,Grocery,5,1200.0
103,Meena,South,Grocery,3,900.0
104,Karthik,East,Electronics,2,60000.0
105,Divya,North,Clothing,4,3200.0
106,Imran,East,Clothing,2,1800.0
107,Sneha,South,Electronics,1,30000.0
108,Arjun,North,Electronics,1,52000.0
109,Kavya,East,Grocery,6,1500.0
110,Rahul,South,Clothing,3,2700.0


root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount: float (nullable = true)



In [0]:
orders.selectExpr("customer","category","amount").display()

customer,category,amount
Asha,Electronics,45000.0
Ravi,Grocery,1200.0
Meena,Grocery,900.0
Karthik,Electronics,60000.0
Divya,Clothing,3200.0
Imran,Clothing,1800.0
Sneha,Electronics,30000.0
Arjun,Electronics,52000.0
Kavya,Grocery,1500.0
Rahul,Clothing,2700.0


In [0]:
orders.filter(col("order_id").between(104,108)).display()

order_id,customer,region,category,qty,amount
104,Karthik,East,Electronics,2,60000.0
105,Divya,North,Clothing,4,3200.0
106,Imran,East,Clothing,2,1800.0
107,Sneha,South,Electronics,1,30000.0
108,Arjun,North,Electronics,1,52000.0


In [0]:
orders.withColumn('region',when(col("order_id") == 105,None).when(col("order_id")==108,None).otherwise('region')).display()
orders.withColumn('amount',when(col("order_id") == 103,None).when(col("order_id")==107,None).otherwise('amount')).display()

order_id,customer,region,category,qty,amount
101,Asha,region,Electronics,1,45000.0
102,Ravi,region,Grocery,5,1200.0
103,Meena,region,Grocery,3,900.0
104,Karthik,region,Electronics,2,60000.0
105,Divya,null,Clothing,4,3200.0
106,Imran,region,Clothing,2,1800.0
107,Sneha,region,Electronics,1,30000.0
108,Arjun,null,Electronics,1,52000.0
109,Kavya,region,Grocery,6,1500.0
110,Rahul,region,Clothing,3,2700.0


order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,amount
102,Ravi,North,Grocery,5,amount
103,Meena,South,Grocery,3,null
104,Karthik,East,Electronics,2,amount
105,Divya,North,Clothing,4,amount
106,Imran,East,Clothing,2,amount
107,Sneha,South,Electronics,1,null
108,Arjun,North,Electronics,1,amount
109,Kavya,East,Grocery,6,amount
110,Rahul,South,Clothing,3,amount


In [0]:
orders = orders.withColumn("range",when(col("amount")>50000,"High")
               .when(col("amount")>10000,"Medium")
               .otherwise("Low"))
orders.display()

order_id,customer,region,category,qty,amount,range
101,Asha,South,Electronics,1,45000.0,Medium
102,Ravi,North,Grocery,5,1200.0,Low
103,Meena,South,Grocery,3,900.0,Low
104,Karthik,East,Electronics,2,60000.0,High
105,Divya,North,Clothing,4,3200.0,Low
106,Imran,East,Clothing,2,1800.0,Low
107,Sneha,South,Electronics,1,30000.0,Medium
108,Arjun,North,Electronics,1,52000.0,High
109,Kavya,East,Grocery,6,1500.0,Low
110,Rahul,South,Clothing,3,2700.0,Low


In [0]:
orders = orders.withColumn("country",lit("India"))
orders.display()

order_id,customer,region,category,qty,amount,range,country
101,Asha,South,Electronics,1,45000.0,Medium,India
102,Ravi,North,Grocery,5,1200.0,Low,India
103,Meena,South,Grocery,3,900.0,Low,India
104,Karthik,East,Electronics,2,60000.0,High,India
105,Divya,North,Clothing,4,3200.0,Low,India
106,Imran,East,Clothing,2,1800.0,Low,India
107,Sneha,South,Electronics,1,30000.0,Medium,India
108,Arjun,North,Electronics,1,52000.0,High,India
109,Kavya,East,Grocery,6,1500.0,Low,India
110,Rahul,South,Clothing,3,2700.0,Low,India


In [0]:
col = [x.upper() for x in orders.columns]
orders = orders.toDF(*col)
orders.display()

ORDER_ID,CUSTOMER,REGION,CATEGORY,QTY,AMOUNT,RANGE,COUNTRY
101,Asha,South,Electronics,1,45000.0,Medium,India
102,Ravi,North,Grocery,5,1200.0,Low,India
103,Meena,South,Grocery,3,900.0,Low,India
104,Karthik,East,Electronics,2,60000.0,High,India
105,Divya,North,Clothing,4,3200.0,Low,India
106,Imran,East,Clothing,2,1800.0,Low,India
107,Sneha,South,Electronics,1,30000.0,Medium,India
108,Arjun,North,Electronics,1,52000.0,High,India
109,Kavya,East,Grocery,6,1500.0,Low,India
110,Rahul,South,Clothing,3,2700.0,Low,India


In [0]:
from pyspark.sql.functions import *

In [0]:
orders = orders.withColumn("EMAIL_ID",lower(concat(col("CUSTOMER"),lit("123@gmail.com"))))
orders.display()



ORDER_ID,CUSTOMER,REGION,CATEGORY,QTY,AMOUNT,RANGE,COUNTRY,EMAIL_ID
101,Asha,South,Electronics,1,45000.0,Medium,India,asha123@gmail.com
102,Ravi,North,Grocery,5,1200.0,Low,India,ravi123@gmail.com
103,Meena,South,Grocery,3,900.0,Low,India,meena123@gmail.com
104,Karthik,East,Electronics,2,60000.0,High,India,karthik123@gmail.com
105,Divya,North,Clothing,4,3200.0,Low,India,divya123@gmail.com
106,Imran,East,Clothing,2,1800.0,Low,India,imran123@gmail.com
107,Sneha,South,Electronics,1,30000.0,Medium,India,sneha123@gmail.com
108,Arjun,North,Electronics,1,52000.0,High,India,arjun123@gmail.com
109,Kavya,East,Grocery,6,1500.0,Low,India,kavya123@gmail.com
110,Rahul,South,Clothing,3,2700.0,Low,India,rahul123@gmail.com


In [0]:
orders.withColumn("MASKED_EMAIL",regexp_replace(col("EMAIL_ID"),"[123]","*")).display()

ORDER_ID,CUSTOMER,REGION,CATEGORY,QTY,AMOUNT,RANGE,COUNTRY,EMAIL_ID,MASKED_EMAIL
101,Asha,South,Electronics,1,45000.0,Medium,India,asha123@gmail.com,asha***@gmail.com
102,Ravi,North,Grocery,5,1200.0,Low,India,ravi123@gmail.com,ravi***@gmail.com
103,Meena,South,Grocery,3,900.0,Low,India,meena123@gmail.com,meena***@gmail.com
104,Karthik,East,Electronics,2,60000.0,High,India,karthik123@gmail.com,karthik***@gmail.com
105,Divya,North,Clothing,4,3200.0,Low,India,divya123@gmail.com,divya***@gmail.com
106,Imran,East,Clothing,2,1800.0,Low,India,imran123@gmail.com,imran***@gmail.com
107,Sneha,South,Electronics,1,30000.0,Medium,India,sneha123@gmail.com,sneha***@gmail.com
108,Arjun,North,Electronics,1,52000.0,High,India,arjun123@gmail.com,arjun***@gmail.com
109,Kavya,East,Grocery,6,1500.0,Low,India,kavya123@gmail.com,kavya***@gmail.com
110,Rahul,South,Clothing,3,2700.0,Low,India,rahul123@gmail.com,rahul***@gmail.com
